In [1]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [2]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
train_data.shape

(14732, 3)

In [7]:
val_data.shape

(818, 3)

In [8]:
# random sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)


# **Pre-Processing**

In [9]:
import re
def clean_data(text):
  text = re.sub(r"\r\n", " ", text) # for line in text
  text = re.sub(r"\s+", " ", text)  # for multiple spaces
  text = re.sub(r"<,*?>", " ", text) # for symbols
  text = text.strip().lower() #remove space and convert in lower case
  return text

In [10]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = train_data["dialogue"].apply(clean_data)
val_data["summary"] = train_data["summary"].apply(clean_data)

## **Tokenize**

In [11]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:
# raw data => tokenized inputs for fine-tuning

def tokenize(data):
  inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
  target = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

  inputs["labels"] = target["input_ids"] #token ids => add to input as labels
  return inputs

In [13]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [14]:
train_dataset[3999]
 # input ids - dialogue => token ids
 # I => EOS, 0=> padding
 # attention mask
 # labels - target => summary token

{'input_ids': [528, 29, 29, 9, 10, 146, 1395, 6, 130, 62, 5741, 12, 608, 8, 829, 1499, 21, 5721, 8665, 3, 2618, 8607, 10, 3, 63, 15, 102, 177, 159, 10, 3, 63, 15, 102, 6, 8, 829, 589, 528, 29, 29, 9, 10, 68, 34, 31, 7, 114, 1283, 1688, 3, 210, 189, 177, 159, 10, 3, 23, 6292, 3654, 227, 335, 1688, 5, 34, 31, 7, 78, 13809, 528, 29, 29, 9, 10, 10191, 6, 3, 23, 278, 31, 17, 317, 1321, 31, 7, 3, 13366, 608, 34, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [15]:
len(train_dataset[0]["input_ids"])

512

In [16]:
type(train_dataset)
type(val_dataset)

list

## Working With Our Model

In [17]:
# NLP => generatio task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [18]:
import torch

if torch.backends.mps.is_available():
  device = torch.device("mps")
elif torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")

print("device:", device)
model.to(device)

device: cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [19]:
training_arg = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
    # 0 => lr default

)

In [20]:
trainer = Trainer(
    model = model,
    args = training_arg,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [21]:
# train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.586243,0.363530
2,0.395991,0.335191
3,0.373225,0.324140
4,0.361157,0.316031
5,0.354601,0.311988
6,0.349701,0.310982


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9034861450195313, metrics={'train_runtime': 1219.633, 'train_samples_per_second': 19.678, 'train_steps_per_second': 2.46, 'total_flos': 3248203235328000.0, 'train_loss': 0.9034861450195313, 'epoch': 6.0})

In [22]:
# model load => fine-tune => save the model

In [23]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [34]:
def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue) #clean

  # tokenize
  inputs = tokenizer(
      dialogue,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt"
  ).to(device)

  # generate the summary => token ids
  model.to(device)
  targets = model.generate(
      input_ids = inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_length = 150,
      num_beams = 4
  )

  # decoded our outputs
  summary = tokenizer.decode(targets[0], skip_special_tokens=True)
  return summary

In [35]:
test_dialogue = """
User: Hi, I'm trying to log into my account portal but I keep getting an 'Error 403: Access Denied' message.
Agent: Hello! I can certainly help you with that. Are you using a corporate VPN right now?
User: Yes, I am connected to our company's secure network.
Agent: Thanks for confirming. Our new security update blocks certain active VPN configurations. Could you temporarily disconnect and try again?
User: Okay, let me turn off the VPN... Hold on.
User: Alright, I disconnected it. Let me refresh the page.
User: Oh, now it's asking for a temporary verification code.
Agent: Perfect, that means you got past the firewall. I just triggered the verification code to your registered email.
User: Let me check my inbox. Yes, I got it. The code is 582-901.
Agent: Great, enter that in and you should be fully redirected to your dashboard.
User: It worked! I'm in now. Thank you so much for the guidance.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary:", summary)

Summary: user is trying to log into his account portal but gets an 'error 403: access denied' message. agent is trying to disconnect and try again. the new security update blocks certain active vpn configurations.


In [36]:
!zip -r saved_summary_model.zip ./saved_summary_model

  adding: saved_summary_model/ (stored 0%)
  adding: saved_summary_model/tokenizer_config.json (deflated 83%)
  adding: saved_summary_model/model.safetensors (deflated 10%)
  adding: saved_summary_model/generation_config.json (deflated 29%)
  adding: saved_summary_model/config.json (deflated 63%)
  adding: saved_summary_model/tokenizer.json (deflated 79%)
